# NIG-TimeNet (PyTorch): the cross-asset model

Data preparation, training and testing of the model that predicts all coins of one time bucket together. Its targets
are 2-D (slots x targets) and it is never flattened: `nig_timenet_torch/model/grid_panel.py` takes the day grid and
returns the day grid. Section 2 builds the training data (skipped when `BUILD = False`), section 3 shows the day grids,
section 4 trains and tests (single split or rolling), section 5 re-trains with a change.

`CHANGES` are dotted keys on `RunSettings`: `panel.*` (group size, relative-to-median targets, slot limits),
`panel_net.*` (width, cross-attention layers), `panel_loss.*` (classification, regression and rank-IC weights).

In [ ]:
TORCH_REPO, TORCH_REF = "yuosef772424/nig-timenet-torch", "feat/native-data-notebooks-uncertainty"
EXPECT_DEVICE = "cuda"                         # "cpu" only for a deliberate CPU run; a cuda run on a CPU VM fails
TOKEN_PATH = "/content/drive/MyDrive/crypto/secrets/github_token.txt"
WORKDIR = "/content/nig_runner"                # clones' scratch, logs, git credentials helper: never /tmp (rule E2)
TORCH_DIR = "/content/nig-timenet-torch"
DRIVE_RUNS = "/content/drive/MyDrive/nig_timenet_runs"
EXTRA_MODULES = (("pyarrow", "pyarrow"), ("pandas_ta_classic", "pandas-ta-classic==0.8.32"), ("requests", "requests"))
RAW_ROOT = "/content/drive/MyDrive"            # holds history_1h/ and crypto_data/asset_registry.csv
DATA_DIR = "/content/drive/MyDrive/crypto_model/preprocessed_data"
DATA_NAME = "preprocessing_output"             # base name (+ _latest) or the full file / folder name
RUN_DIR = "/content/drive/MyDrive/nig_timenet_runs/cross_asset"
BUILD = False                                  # True: build the data first (DATA_PRESETS + DATA_CHANGES), then train on it
DATA_PRESETS = ("hourly_w32_s8",)
DATA_CHANGES = {}                              # dotted keys of data/settings.py, e.g. {"windows.stride": 4}
CHANGES = {"kind": "panel", "train.epochs": 30, "train.batch_size": 16, "panel.relative": "median"}
ROLLING = {}                                   # e.g. {"train_span": "365D", "val_span": "30D", "test_span": "30D", "step": "30D", "embargo": "4h", "init": "warm"}
RETRAIN = {}                                   # e.g. {"panel_loss.w_ic": 1.0}; empty = no second run

## 1. Setup
The first cell is plain code: nothing of the package exists on the VM yet, so it downloads the one stdlib-only file
`colab/bootstrap.py` with the token and runs it. Everything after it is a normal import.

In [ ]:
import getpass, os, urllib.error, urllib.request
from google.colab import drive
drive.mount("/content/drive")
token = open(TOKEN_PATH).read().strip() if os.path.exists(TOKEN_PATH) else getpass.getpass("GitHub token (not shown): ")
url = f"https://api.github.com/repos/{TORCH_REPO}/contents/src/nig_timenet_torch/colab/bootstrap.py?ref={TORCH_REF}"
req = urllib.request.Request(url, headers={"Authorization": "Bearer " + token.strip(), "User-Agent": "nig-colab",
                                           "Accept": "application/vnd.github.raw"})
try:
    boot_source = urllib.request.urlopen(req, timeout=60).read().decode()
except urllib.error.HTTPError as e:
    raise SystemExit(f"GitHub answered HTTP {e.code}: the token is missing, expired or cannot read {TORCH_REPO}. "
                     f"Delete {TOKEN_PATH} and run this cell again.")
boot = {}
exec(compile(boot_source, "bootstrap.py", "exec"), boot)
token = boot["acquire_token"](TOKEN_PATH, TORCH_REPO, lambda: getpass.getpass("GitHub token (not shown): "))

In [ ]:
shas = boot["setup_clones"](token, WORKDIR, {"torch": (TORCH_REPO, TORCH_REF, TORCH_DIR)})
install = boot["install_package"](TORCH_DIR, modules=EXTRA_MODULES)

In [ ]:
from nig_timenet_torch.colab.session import check_environment
setup = check_environment(EXPECT_DEVICE, shas, install, WORKDIR, DRIVE_RUNS)

## 2. Data
With `BUILD` the data is built from the raw files and written as `<NAME>_<stamp>.dataset` plus `_latest`; otherwise the
existing file named by `DATA_NAME` is used.

In [ ]:
from nig_timenet_torch.data.pipeline import build_artifact
from nig_timenet_torch.data.presets import preset
data_settings = preset(*DATA_PRESETS).updated({"source.root": RAW_ROOT, "output.directory": DATA_DIR, "output.name": DATA_NAME, **DATA_CHANGES})
manifest = build_artifact(data_settings)[1] if BUILD else None
print(manifest["export"]["rows"] if manifest else "using the existing file", "rows")

In [ ]:
from nig_timenet_torch.core.config import updated
from nig_timenet_torch.data_io.loading import load_dataset
from nig_timenet_torch.workflow.settings import RunSettings
paths = {"data.directory": DATA_DIR, "data.name": DATA_NAME, "run_dir": RUN_DIR}
settings = updated(RunSettings(), {**paths, **CHANGES, **({"rolling": ROLLING} if ROLLING else {})})
loaded = load_dataset(settings.data)
print(loaded.path, loaded.format, len(loaded.dataset["last_candles"]), "windows")

## 3. The day grids
The train part as the model sees it: `days x slots` rows with a mask, and targets `days x slots x 3` (high, low, close)
relative to the day median.

In [ ]:
from nig_timenet_torch.data.panel import build_panel
from nig_timenet_torch.workflow.run_common import make_plan
plan = make_plan(loaded.dataset, settings)
grid = build_panel(loaded.dataset, plan.train_idx, settings.panel)
print("days x slots:", grid.rows.shape, "| real rows:", int(grid.mask.sum()), "| targets:", grid.y_reg.shape)

## 4. Train and test

In [ ]:
from nig_timenet_torch.workflow.run import run
result = run(settings, loaded)

In [ ]:
from nig_timenet_torch.workflow.run import rolling_summary, scalars
print(rolling_summary(result) if settings.rolling else scalars(result.report))

## 5. Re-train with a change

In [ ]:
from nig_timenet_torch.workflow.run import retrain
second = retrain(result, RETRAIN) if RETRAIN else None
print(scalars(second.report) if second and second.report else "no re-train requested")